# Comparação de 10 Estratégias de Chunking
## LangChain Text Splitters + Busca Semântica via OpenRouter

Este notebook compara 10 estratégias de divisão de texto (*chunking*) para avaliar como o tamanho, overlap e tipo de divisão impactam a qualidade da busca semântica.

---
### 1. Instalação de Bibliotecas

In [ ]:
!pip install -q openai langchain-text-splitters matplotlib pandas

---
### 2. Configuração e Cliente API

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata
from google.colab import files
from openai import OpenAI

# Configuração do cliente OpenAI via OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_API_KEY')
)

MAX_CHARS = 6000
EMBEDDING_MODEL = 'openai/text-embedding-3-small'

QUERIES = [
    'O que e autonomia e opacidade algoritmica?',
    'O que e o diario de bordo da IA?',
]

print('Configuração concluída com sucesso!')

---
### 3. Upload dos Arquivos Markdown (.md)

In [ ]:
print("Faça upload dos arquivos .md da Aula 02:")
uploaded = files.upload()

documentos = []
for nome_arquivo, conteudo in uploaded.items():
    texto = conteudo.decode('utf-8')
    documentos.append(texto)
    print(f"Arquivo carregado: {nome_arquivo} ({len(conteudo)} bytes)")

texto_completo = "\n\n".join(documentos)
print(f"\nTotal: {len(documentos)} arquivos, {len(texto_completo)} caracteres")

---
### 4. Funções Auxiliares

In [ ]:
def get_embedding(texto):
    """Gera embedding de um texto via OpenRouter."""
    texto = str(texto)[:MAX_CHARS]
    resposta = client.embeddings.create(
        input=texto,
        model=EMBEDDING_MODEL,
    )
    return resposta.data[0].embedding

def similaridade_cosseno(vec_a, vec_b):
    """Calcula similaridade de cosseno entre dois vetores."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(dot / (norm_a * norm_b))

def busca_semantica(query, chunks, top_k=3):
    """Realiza busca semântica nos chunks."""
    vec_query = get_embedding(query)
    resultados = []
    batch_size = 10
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        batch_truncado = [str(c)[:MAX_CHARS] for c in batch]
        resposta = client.embeddings.create(
            input=batch_truncado,
            model=EMBEDDING_MODEL,
        )
        for j, dado in enumerate(resposta.data):
            sim = similaridade_cosseno(vec_query, dado.embedding)
            resultados.append({"Trecho": batch[j], "Similaridade": sim})
        if i + batch_size < len(chunks):
            time.sleep(1)
    resultados.sort(key=lambda x: x["Similaridade"], reverse=True)
    return resultados[:top_k]

def estatisticas_chunks(chunks):
    tamanhos = [len(str(c)) for c in chunks]
    if not tamanhos:
        return {"total": 0, "media": 0, "min": 0, "max": 0}
    return {
        "total": len(chunks),
        "media": int(np.mean(tamanhos)),
        "min": min(tamanhos),
        "max": max(tamanhos),
    }

print('Funções auxiliares prontas!')

---
### 5. Definição das 10 Estratégias de Chunking

In [ ]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

estrategias = [
    {
        'grupo': 1,
        'nome': 'Fixo 200 chars, sem overlap',
        'variavel': 'tamanho (extremo baixo)',
        'splitter': CharacterTextSplitter(separator='', chunk_size=200, chunk_overlap=0),
        'tipo': 'texto',
    },
    {
        'grupo': 2,
        'nome': 'Fixo 500 chars, sem overlap',
        'variavel': 'tamanho',
        'splitter': CharacterTextSplitter(separator='', chunk_size=500, chunk_overlap=0),
        'tipo': 'texto',
    },
    {
        'grupo': 3,
        'nome': 'Fixo 1000 chars, sem overlap',
        'variavel': 'tamanho',
        'splitter': CharacterTextSplitter(separator='', chunk_size=1000, chunk_overlap=0),
        'tipo': 'texto',
    },
    {
        'grupo': 4,
        'nome': 'Fixo 2000 chars, sem overlap',
        'variavel': 'tamanho (extremo alto)',
        'splitter': CharacterTextSplitter(separator='', chunk_size=2000, chunk_overlap=0),
        'tipo': 'texto',
    },
    {
        'grupo': 5,
        'nome': 'Fixo 500, overlap 50 (10%)',
        'variavel': 'overlap leve',
        'splitter': CharacterTextSplitter(separator='', chunk_size=500, chunk_overlap=50),
        'tipo': 'texto',
    },
    {
        'grupo': 6,
        'nome': 'Fixo 500, overlap 200 (40%)',
        'variavel': 'overlap pesado',
        'splitter': CharacterTextSplitter(separator='', chunk_size=500, chunk_overlap=200),
        'tipo': 'texto',
    },
    {
        'grupo': 7,
        'nome': 'Recursivo 500, overlap 50',
        'variavel': 'estratégia recursiva leve',
        'splitter': RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50),
        'tipo': 'texto',
    },
    {
        'grupo': 8,
        'nome': 'Recursivo 1000, overlap 100',
        'variavel': 'estratégia recursiva moderada',
        'splitter': RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100),
        'tipo': 'texto',
    },
    {
        'grupo': 9,
        'nome': 'Recursivo 500, overlap 100',
        'variavel': 'estratégia recursiva pesada',
        'splitter': RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100),
        'tipo': 'texto',
    },
    {
        'grupo': 10,
        'nome': 'Por seção/heading Markdown',
        'variavel': 'estrutura semântica',
        'splitter': MarkdownHeaderTextSplitter(headers_to_split_on=[('#', 'Header 1'), ('##', 'Header 2'), ('###', 'Header 3')]),
        'tipo': 'markdown',
    },
]

print(f'{len(estrategias)} estratégias definidas!')

---
### 6. Execução e Comparação das Estratégias

In [ ]:
resultados_finais = []

for est in estrategias:
    grupo = est['grupo']
    nome = est['nome']
    variavel = est['variavel']
    splitter = est['splitter']

    print('=' * 70)
    print(f'GRUPO {grupo}: {nome}')
    print('=' * 70)

    try:
        if est['tipo'] == 'markdown':
            chunks_docs = splitter.split_text(texto_completo)
            chunks = [doc.page_content for doc in chunks_docs if doc.page_content.strip()]
        else:
            chunks = splitter.split_text(texto_completo)
            chunks = [c for c in chunks if c.strip()]
    except Exception as e:
        print(f'Erro ao dividir texto: {e}')
        continue

    chunks = [c for c in chunks if len(c.strip()) >= 10]
    stats = estatisticas_chunks(chunks)
    print(f'Chunks gerados: {stats["total"]} (tamanho médio: {stats["media"]} chars)')

    melhores_sims = []
    for qi, query in enumerate(QUERIES, 1):
        print(f'  Query {qi}: "{query}"')
        try:
            top_resultados = busca_semantica(query, chunks, top_k=3)
            melhor_sim = top_resultados[0]['Similaridade'] if top_resultados else 0.0
            melhores_sims.append(melhor_sim)
            for idx, r in enumerate(top_resultados, 1):
                trecho_curto = r['Trecho'][:120].replace('\n', ' ')
                print(f'    TOP {idx} (Sim: {r["Similaridade"]:.4f}): {trecho_curto}...')
        except Exception as e:
            print(f'    Erro na busca: {e}')
            melhores_sims.append(0.0)

    resultados_finais.append({
        'Grupo': grupo,
        'Estrategia': nome,
        'Variavel': variavel,
        'Chunks': stats['total'],
        'Tam. Medio': stats['media'],
        'Sim Q1': round(melhores_sims[0], 4) if len(melhores_sims) > 0 else 0.0,
        'Sim Q2': round(melhores_sims[1], 4) if len(melhores_sims) > 1 else 0.0,
        'Media': round(np.mean(melhores_sims), 4) if melhores_sims else 0.0,
    })

print('\nTodas as estratégias foram executadas com sucesso!')

---
### 7. Tabela Comparativa Final

In [ ]:
df = pd.DataFrame(resultados_finais)
df = df.sort_values('Media', ascending=False)
df

---
### 8. Gráfico Comparativo das Estratégias

In [ ]:
df_plot = pd.DataFrame(resultados_finais).sort_values('Grupo')

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(df_plot))
largura = 0.25

ax.bar(x - largura, df_plot['Sim Q1'], largura, label='Query 1 (autonomia)', color='#4285F4')
ax.bar(x, df_plot['Sim Q2'], largura, label='Query 2 (diário)', color='#EA4335')
ax.bar(x + largura, df_plot['Media'], largura, label='Média', color='#34A853')

ax.set_xlabel('Estratégia de Chunking')
ax.set_ylabel('Similaridade Cosseno')
ax.set_title('Comparação de 10 Estratégias de Chunking - Busca Semântica')
ax.set_xticks(x)
ax.set_xticklabels([f'G{int(g)}' for g in df_plot['Grupo']])
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()